# Task B — SimCLR Pretraining
### CSE 438 Part B — SSL for Semantic Segmentation

**Pretext task:** maximize agreement between two augmented views of the same
image vs. all other images in the batch (NT-Xent loss). No class labels used.

**Architecture note:** the encoder exposes BOTH a pooled vector (used only
here, for the contrastive loss) and the pre-pool spatial feature map
(`[2048, 7, 7]`), which Task C will attach the DeepLabV3 ASPP decoder to.
Only the pre-pool feature map survives into Task C/D — the projection head
here is discarded after pretraining, standard SimCLR practice.

In [1]:
import os, time, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


Using device: cuda


## Config cell — report these in Section 6.4/6.5

In [2]:
CONFIG = {
    "method": "SimCLR",
    "backbone": "resnet50",
    "checkpoint_source": "torchvision_imagenet",  # continuing from ImageNet weights (recommended default)
    "pretrain_epochs": 5,
    "batch_size": 32,           # reduce if OOM on Kaggle T4/P100 (16GB)
    "img_size": 224,
    "projection_dim": 128,
    "temperature": 0.5,           # NT-Xent temperature -- key hyperparameter to report
    "optimizer": "Adam",
    "lr": 3e-4,
    "weight_decay": 1e-6,
    "seed": 42,
}

random.seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"]); torch.cuda.manual_seed_all(CONFIG["seed"])

print(json.dumps(CONFIG, indent=2))


{
  "method": "SimCLR",
  "backbone": "resnet50",
  "checkpoint_source": "torchvision_imagenet",
  "pretrain_epochs": 5,
  "batch_size": 32,
  "img_size": 224,
  "projection_dim": 128,
  "temperature": 0.5,
  "optimizer": "Adam",
  "lr": 0.0003,
  "weight_decay": 1e-06,
  "seed": 42
}


## Load the verified split artifact from the Data Prep notebook

Attach the data-prep dataset as an input to this notebook, then point the path below at it.

In [3]:
TRAIN_CSV = "/kaggle/input/notebooks/mashrur135/nb-01/artifacts/splits/train_split.csv"

train_df = pd.read_csv(TRAIN_CSV)
print(f"Unlabelled pretraining images: {len(train_df)}")


Unlabelled pretraining images: 280


## SimCLR two-view augmentation

- `RandomResizedCrop`: forces recognizing the same scene at different scales/positions
- `ColorJitter` + `RandomGrayscale`: forces color invariance, prevents shortcut learning via color histograms
- Vertical flip + 90° rotation included: this is nadir aerial imagery, which has no fixed "up", unlike street-level photos

In [4]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

simclr_transform = T.Compose([
    T.RandomResizedCrop(CONFIG["img_size"], scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomApply([T.RandomRotation(90)], p=0.5),
    T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class SimCLRDataset(Dataset):
    def __init__(self, df, transform):
        self.paths = df["image_path"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img), self.transform(img)  # view1, view2

train_ds = SimCLRDataset(train_df, simclr_transform)
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
                           num_workers=2, drop_last=True, pin_memory=True)
print(f"Batches per epoch: {len(train_loader)}")


Batches per epoch: 8


## Encoder — ResNet-50 exposing BOTH pooled vector and spatial feature map

This split is the key architectural choice: everything before the final
pooling layer (`backbone`) feeds Task C/D later; the pooling+projector path
below is only used for this pretraining loss and gets discarded afterwards.

In [5]:
class SimCLREncoder(nn.Module):
    def __init__(self, projection_dim=128, pretrained=True):
        super().__init__()
        resnet = models.resnet50(weights="IMAGENET1K_V2" if pretrained else None)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])  # -> [B, 2048, 7, 7]
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.projector = nn.Sequential(
            nn.Linear(2048, 2048),
            nn.ReLU(inplace=True),
            nn.Linear(2048, projection_dim),
        )

    def forward(self, x):
        feature_map = self.backbone(x)               # [B, 2048, 7, 7] -- used by Task C/D
        pooled = self.pool(feature_map).flatten(1)     # [B, 2048]
        z = self.projector(pooled)                     # [B, projection_dim]
        z = F.normalize(z, dim=1)
        return feature_map, z

encoder = SimCLREncoder(projection_dim=CONFIG["projection_dim"]).to(DEVICE)

with torch.no_grad():
    dummy = torch.randn(2, 3, CONFIG["img_size"], CONFIG["img_size"]).to(DEVICE)
    fmap, z = encoder(dummy)
    print(f"feature_map shape: {fmap.shape}")   # expect [2, 2048, 7, 7]
    print(f"projection z shape: {z.shape}")     # expect [2, 128]


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 157MB/s] 


feature_map shape: torch.Size([2, 2048, 7, 7])
projection z shape: torch.Size([2, 128])


## NT-Xent loss

For N images -> 2N views. Each view's positive is the OTHER view of the same
image; every other view (2N-2 of them) is a negative. Lower temperature =
sharper separation between positives and negatives.

In [6]:
def nt_xent_loss(z1, z2, temperature=0.5):
    batch_size = z1.shape[0]
    z = torch.cat([z1, z2], dim=0)  # [2B, D]
    sim_matrix = torch.matmul(z, z.T) / temperature  # [2B, 2B]

    mask = torch.eye(2 * batch_size, device=z.device, dtype=torch.bool)
    sim_matrix.masked_fill_(mask, -1e9)

    positive_idx = torch.cat([
        torch.arange(batch_size, 2 * batch_size),
        torch.arange(0, batch_size)
    ]).to(z.device)

    return F.cross_entropy(sim_matrix, positive_idx)


## Training loop (50 epochs, fixed requirement)

In [ ]:
optimizer = torch.optim.Adam(encoder.parameters(), lr=CONFIG["lr"],
                              weight_decay=CONFIG["weight_decay"])

loss_history, epoch_time_history = [], []

for epoch in range(CONFIG["pretrain_epochs"]):
    t0 = time.time()
    encoder.train()
    running_loss = 0.0

    for view1, view2 in train_loader:
        view1, view2 = view1.to(DEVICE), view2.to(DEVICE)
        _, z1 = encoder(view1)
        _, z2 = encoder(view2)
        loss = nt_xent_loss(z1, z2, temperature=CONFIG["temperature"])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    epoch_time = time.time() - t0
    loss_history.append(epoch_loss)
    epoch_time_history.append(epoch_time)

    print(f"Epoch {epoch+1}/{CONFIG['pretrain_epochs']} | NT-Xent loss: {epoch_loss:.4f} | time: {epoch_time:.1f}s")
    # If per-epoch time exceeds ~10 min, split pretraining and downstream into two notebooks (per assignment).


Epoch 1/5 | NT-Xent loss: 3.5389 | time: 69.1s


## Loss curve + per-epoch time (required for report)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(loss_history) + 1), loss_history)
plt.xlabel("Epoch"); plt.ylabel("NT-Xent Loss"); plt.title("SimCLR Pretraining Loss Curve")
plt.grid(alpha=0.3)
plt.savefig("/kaggle/working/simclr_pretrain_loss.png", dpi=150, bbox_inches="tight")
plt.show()

avg_epoch_time = sum(epoch_time_history) / len(epoch_time_history)
print(f"Average per-epoch time: {avg_epoch_time:.1f}s")


## Save encoder checkpoint for Task C (projection head discarded)

In [ ]:
torch.save({
    "backbone_state_dict": encoder.backbone.state_dict(),
    "config": CONFIG,
    "final_loss": loss_history[-1],
    "avg_epoch_time_sec": avg_epoch_time,
}, "/kaggle/working/simclr_encoder_checkpoint.pth")

print("Saved: /kaggle/working/simclr_encoder_checkpoint.pth")
print("Feeds Task C: attach ASPP decoder to encoder.backbone's [2048,7,7] output.")
